# 02 — Exact Matching

Exact matching is the first building block: rows match when the values in the rule columns are **equal** (after standardization). We **start with raw** data and **standardize** it explicitly (same pipeline as 01), then match. So: raw → clean → match—no hidden steps. Here we run a single rule (`email_clean`), then *cascading* rules, and see how to evaluate results. The next notebook makes the **measurement loop** explicit.

**Back:** [01 Preparation](01_preparation.ipynb) · **Next:** [03 The Measurement Loop](03_measurement_loop.ipynb)

## 1. Load raw data, standardize, then match

We start with **raw** data (same as 01: column name mismatches, nulls, messy values), then **explicitly** standardize it so we have clean tables for matching. No magic: load raw → standardize → match.

**Data:** 500 left, 500 right, **40 known pairs**. After standardization we match on `email_clean` and optionally cascade to `full_name`.

In [1]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution, standardize_for_matching

# 1. Load raw (same as 01: right has email_address, postal_code; messy values)
left_raw, right_raw, ground_truth = load_entity_resolution()
print("Raw: right columns", right_raw.columns[:4], "...")

# 2. Standardize: align schema, add email_clean and full_name
left, right = standardize_for_matching(left_raw, right_raw)
print("Clean: same columns?", set(left.columns) == set(right.columns))

# 3. Match on clean data
matcher = Matcher(
    left=left,
    right=right,
    left_id="id",
    right_id="id",
)
n_left, n_right = left.shape[0], right.shape[0]
n_gt = ground_truth.shape[0]
print(f"Left: {n_left} rows, Right: {n_right} rows, Ground truth: {n_gt} pairs")

Raw: right columns ['id', 'email_address', 'first_name', 'last_name'] ...
Clean: same columns? True
Left: 500 rows, Right: 500 rows, Ground truth: 40 pairs


## 2. Single rule vs cascading rules

Start with **one rule** and see how many pairs we find. Then **add a second rule** (cascade): matcher tries the first rule, then for rows that didn't match it tries the second. Each step gives incrementally better recall.

**Step 2a:** Match on `email_clean` only. We find the pairs that share the same (cleaned) email.

In [2]:
results_email = matcher.match(rules="email_clean")
m_email = results_email.evaluate(
    ground_truth,
    left_id_col="id",
    right_id_col="id_right",
)
print(f"Matches: {results_email.count}")
tp, total = m_email["true_positives"], ground_truth.shape[0]
print(f"Recall: {m_email['recall']:.2%} (found {tp} of {total} true pairs)")

Matches: 20
Recall: 50.00% (found 20 of 40 true pairs)


**Step 2b:** Add `full_name` as a **cascade** (second rule). Use a list of rules: `["email_clean", ["full_name"]]`. Each rule is tried in order; the second runs only on left rows that didn't match the first. (A single list like `["email_clean", "full_name"]` would mean one rule: match when *both* fields match—AND, not cascade.)

In [3]:
# Cascade: second rule must be a list so matcher treats it as "then try full_name for unmatched"
results_cascade = matcher.match(rules=[
    "email_clean",
    ["full_name"],
])
m_cascade = results_cascade.evaluate(
    ground_truth,
    left_id_col="id",
    right_id_col="id_right",
)
print(f"Matches: {results_cascade.count}")
tp, total = m_cascade["true_positives"], ground_truth.shape[0]
print(f"Recall: {m_cascade['recall']:.2%} (found {tp} of {total} true pairs)")

Matches: 31
Recall: 75.00% (found 30 of 40 true pairs)


## 3. Compare the two runs

We already evaluated each run above. Here's the full picture side by side: precision, recall, F1. The cascade adds recall (more true pairs found) while we keep an eye on precision. In the next notebook we'll turn this into a repeatable **measurement loop**.

In [4]:
comparison = pl.DataFrame({
    "run": ["email_clean only", "email_clean+name"],
    "precision": [f"{m_email['precision']:.2%}", f"{m_cascade['precision']:.2%}"],
    "recall": [f"{m_email['recall']:.2%}", f"{m_cascade['recall']:.2%}"],
    "f1": [f"{m_email['f1']:.2%}", f"{m_cascade['f1']:.2%}"],
})
display(comparison)

run,precision,recall,f1
str,str,str,str
"""email_clean only""","""100.00%""","""50.00%""","""66.67%"""
"""email_clean+name""","""96.77%""","""75.00%""","""84.51%"""


You've seen the evaluate API. Next we'll discuss how to **use these evaluation metrics** and **understand the tradeoffs** (e.g. when to favour recall vs precision, how to tune). That's [03 The Measurement Loop](03_measurement_loop.ipynb)—measure → change one thing → measure again → compare. After that we add fuzzy matching (04).